# FLChain Layer Compression: 12-Layer Curves for DR-TabPFN (Work 8, FLChain)

This notebook measures, at each of the 12 DR-TabPFN layer outputs, three things across depth: how the target-token embeddings spread, how well a linear probe decodes 3-year death, and how the gradient of the death probability changes. The data, splits, scaling and fitting seed are those of the Work 6b FLChain notebook (`flchain_probe.ipynb`). Its layer-11 values are the reference that this notebook must reproduce.

- **Primary target:** uncensored 3-year fixed-horizon mortality (`H = 1095` days).
- **Scaling:** consistent context-based scaling (`ContextMinMaxScaler`).
- **Evaluation strata:** `1997_eval` (held-out part of 1997) and `1998-2003` (later years). Fitting rows for the scale and the probe are the `ref_fit` rows.
- **Descriptive only:** the notebook reports curves with bootstrap intervals and the model's own AUROC. It does not choose a layer.

## 1. Imports
Libraries used by the steps below. The helper functions are in `layer_compression.py`, `flchain_probe.py` and `concept_subgroup_drift_pilot.py`.

In [1]:
import json
import os
import resource
import subprocess
import time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import concept_subgroup_drift_pilot as pilot
import flchain_probe as probe
import layer_compression as lc
import tcav_geometry as geometry

T0 = time.monotonic()
STARTED_UTC = datetime.now(timezone.utc).isoformat()
STEP_SECONDS = {}

## 2. Constants
One cell holds every setting. The dry run before the full run edits only `EVAL_ROWS_PER_STRATUM` (and the bootstrap draw count). Seeds: fit and split `42` (as Work 6b), bootstrap `20261106`, random directions `20261124` (fixed inside `flchain_probe`). Tolerances are the engineering tolerances of the plan. The output folder follows `$RPI_OUT_DIR`, as in Work 6b.

In [2]:
CSV_PATH = Path("data/flchain/free_light_chain_mortality.csv")
WORK6B_DIR = Path("stats/flchain_last_layer_probe")
OUT_DIR = Path(os.environ.get("RPI_OUT_DIR", "stats/flchain_layer_compression"))
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

BUILD = "horizon_1095d"
SCALING = "consistent"
HORIZON_DAYS = 1095
DEVICE = "cuda"
FIT_SEED = 42
BOOTSTRAP_SEED = 20261106
DIRECTION_SEED = 20261124
FD_DIRECTION_SEED = 42
BOOTSTRAP_DRAWS = 500
EVAL_ROWS_PER_STRATUM = None
CAPTURE_BATCH_ROWS = 256
REPLAY_BATCH_ROWS = 256
FD_ROWS = 8
FORWARD_TOLERANCE = {"rtol": 1e-4, "atol": 1e-5}
GRADIENT_TOLERANCE = {"rtol": 1e-3, "atol": 1e-5}

assert HORIZON_DAYS == probe.PRIMARY_HORIZON_DAYS
assert DIRECTION_SEED == probe.DIRECTION_SEED
if not torch.cuda.is_available():
    raise RuntimeError("DEVICE is cuda but CUDA is unavailable on this host")
GPU_NAME = torch.cuda.get_device_name(0)
torch.cuda.reset_peak_memory_stats()

checks = []


def record_check(check, measured, rule, passed, scaling="none"):
    checks.append({"check": check, "build": BUILD, "scaling": scaling, "measured": measured,
                   "rule": rule, "passed": bool(passed)})


def write_checks_and_stop_on_failure():
    checks_table = pd.DataFrame(checks)
    checks_table.to_csv(OUT_DIR / "checks.csv", index=False)
    failed = checks_table.loc[~checks_table["passed"], "check"].tolist()
    if failed:
        raise RuntimeError(f"Engineering checks failed: {failed}")
    return checks_table


print(f"Device: {DEVICE} ({GPU_NAME}) | Out: {OUT_DIR} | Torch: {torch.__version__}")
print(f"Evaluation rows per stratum: {'all' if EVAL_ROWS_PER_STRATUM is None else EVAL_ROWS_PER_STRATUM}")

Device: cuda (NVIDIA GeForce RTX 2080) | Out: /home/rafaelgomes/SAE-TCAV/stats/flchain_layer_compression | Torch: 2.1.2+cu121
Evaluation rows per stratum: all


## 3. Input Integrity and Chronological Row Sets
Check the FLChain CSV against the pinned SHA-256, rebuild the 3-year target and the Work 6b chronological split, and confirm that the context, fitting and evaluation rows are disjoint.

In [3]:
step_start = time.monotonic()
csv_sha256 = probe.file_sha256(CSV_PATH)
record_check("flchain_csv_sha256_matches_work6b", csv_sha256, probe.EXPECTED_CSV_SHA256,
             csv_sha256 == probe.EXPECTED_CSV_SHA256)
df_raw = probe.load_flchain_dataframe(CSV_PATH, expected_sha256=probe.EXPECTED_CSV_SHA256)
kept_df, y = probe.build_flchain_target(df_raw, HORIZON_DAYS)
split = probe.split_flchain_chronological(kept_df, y)
X = probe.extract_feature_matrix(kept_df)
years = kept_df["sample.yr"].to_numpy(dtype=np.int64)
subject_ids = kept_df["subject_id"].to_numpy(dtype=np.int64)

context_idx = split["context_idx"]
ref_fit_idx = split["ref_fit_idx"]
ref_eval_idx = split["ref_eval_idx"]
future_idx = split["future_idx"]
covered = np.concatenate([context_idx, ref_fit_idx, ref_eval_idx, future_idx])
record_check("chronological_split_is_partition", len(kept_df),
             "every kept row in exactly one of context, ref_fit, ref_eval, future",
             len(np.unique(covered)) == len(kept_df) == len(covered))

strata_idx = {"1997_eval": split["strata"]["1997_eval"], "1998-2003": split["strata"]["1998-2003"]}
full_stratum_sizes = {name: len(idx) for name, idx in strata_idx.items()}
if EVAL_ROWS_PER_STRATUM is not None:
    strata_idx = {name: idx[:EVAL_ROWS_PER_STRATUM] for name, idx in strata_idx.items()}
eval_idx = np.concatenate(list(strata_idx.values()))
stratum_rows = {}
start = 0
for name, idx in strata_idx.items():
    stratum_rows[name] = slice(start, start + len(idx))
    start += len(idx)

lc.check_disjoint_rows(context_idx, ref_fit_idx, eval_idx)
record_check("context_fitting_evaluation_rows_disjoint", 0,
             "no row shared between context, ref_fit and evaluation (check_disjoint_rows)", True)

row_counts = pd.DataFrame([
    {"set": name, "rows": len(idx), "deaths": int(y[idx].sum())}
    for name, idx in {"context": context_idx, "ref_fit": ref_fit_idx, "ref_eval": ref_eval_idx,
                      "future": future_idx, "evaluated_1997_eval": strata_idx["1997_eval"],
                      "evaluated_1998-2003": strata_idx["1998-2003"]}.items()
])
print(row_counts.to_string(index=False))
STEP_SECONDS["inputs"] = round(time.monotonic() - step_start, 2)

                set  rows  deaths
            context  5390     433
            ref_fit   340      27
           ref_eval   340      26
             future  1673     120
evaluated_1997_eval   340      26
evaluated_1998-2003  1673     120


## 4. Fit the Context Model (consistent scaling)
Fit DR-TabPFN on the `1995-1997` trailing context rows with the Work 6b call. Read the token count `T` from the fitted model, and record the identity of the fitted cached model so that it can be compared again after replay.

In [4]:
step_start = time.monotonic()
fitted = probe.fit_dr_tabpfn_flchain(
    X[context_idx], y[context_idx], years[context_idx],
    device=DEVICE, seed=FIT_SEED, domain_origin=probe.DOMAIN_ORIGIN, use_context_scaler=True,
)
classifier = fitted["classifier"]
TOKEN_COUNT = lc.token_count(classifier)
identity_before = pilot.predictor_context_identity(classifier)
record_check("context_rows_used_without_subsampling", fitted["context_rows"],
             "context_rows == subsample_rows == number of context rows",
             fitted["subsample_rows"] == fitted["context_rows"] == len(context_idx), scaling=SCALING)
print(f"Tokens per row T = {TOKEN_COUNT} | context rows = {fitted['context_rows']} | "
      f"fit device = {fitted['device']}")
STEP_SECONDS["fit"] = round(time.monotonic() - step_start, 2)

Initialized decoder for standard with (None, 10)  and nout 10


Tokens per row T = 19 | context rows = 5390 | fit device = cuda


## 5. Capture All 12 Layers
One capture call per set of rows. The fitting rows give the target and feature-mean embeddings for the scale and the probe. The evaluation rows also keep the layer-0 output (the replay start), the death probability and the head continuation.

In [5]:
step_start = time.monotonic()
fit_capture = lc.capture_layers(fitted, X[ref_fit_idx], years[ref_fit_idx], batch_size=CAPTURE_BATCH_ROWS)
eval_capture = lc.capture_layers(fitted, X[eval_idx], years[eval_idx], batch_size=CAPTURE_BATCH_ROWS)
LAYER_COUNT = eval_capture["target"].shape[1]
LAST_LAYER = LAYER_COUNT - 1
ref_fit_target = fit_capture["target"]
ref_fit_feature_mean = fit_capture["feature_mean"]
eval_target = eval_capture["target"]
eval_feature_mean = eval_capture["feature_mean"]
eval_probability = eval_capture["probabilities"][:, 1]
shapes_ok = (ref_fit_target.shape == (len(ref_fit_idx), LAYER_COUNT, 192)
             and eval_target.shape == (len(eval_idx), LAYER_COUNT, 192)
             and tuple(eval_capture["layer0"].shape) == (len(eval_idx), TOKEN_COUNT, 192))
record_check("capture_shapes_all_layers", LAYER_COUNT,
             "target and feature mean (rows, layers, 192); layer 0 full (rows, T, 192)", shapes_ok, scaling=SCALING)
print(f"Layers captured: {LAYER_COUNT} | ref_fit rows: {len(ref_fit_idx)} | evaluation rows: {len(eval_idx)}")
STEP_SECONDS["capture"] = round(time.monotonic() - step_start, 2)

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Layers captured: 12 | ref_fit rows: 340 | evaluation rows: 2013


## 6. Replay Layers 1..11 and Take Gradients at Every Layer
The cached model cannot be differentiated, so the layers are recomputed from copies of the weights and the head-0 context keys and values. One backward pass of the summed death probability gives the target-token gradient at every layer. Evaluation rows are processed in batches; rows do not interact, so batch size does not change the result.

In [6]:
step_start = time.monotonic()
weights = lc.clone_replay_weights(classifier)
continuation = eval_capture["continuation"]
gradient_chunks, probability_chunks = [], []
for start in range(0, len(eval_idx), REPLAY_BATCH_ROWS):
    first_layer = eval_capture["layer0"][start:start + REPLAY_BATCH_ROWS]
    probability, gradients = lc.layer_gradients(weights, continuation, first_layer)
    gradient_chunks.append(torch.stack(gradients, dim=1).cpu().numpy())
    probability_chunks.append(probability.cpu().numpy())
eval_gradients = np.concatenate(gradient_chunks)
replay_probability = np.concatenate(probability_chunks)
print(f"Gradient array: {eval_gradients.shape} (rows, layers, 192)")
STEP_SECONDS["replay_gradients"] = round(time.monotonic() - step_start, 2)

Gradient array: (2013, 12, 192) (rows, layers, 192)


## 7. Replay Parity With the Cached Forward and Finite-Difference Checks
Per layer, the replayed target token and feature mean must equal the captured ones, and the replayed probability must equal the captured probability. Then a central finite-difference check, in float64 on the first `8` evaluation rows, tests the gradient of every layer.

In [7]:
step_start = time.monotonic()
replay_target_chunks, replay_mean_chunks, replay_prob_chunks = [], [], []
with torch.no_grad():
    for start in range(0, len(eval_idx), REPLAY_BATCH_ROWS):
        first_layer = eval_capture["layer0"][start:start + REPLAY_BATCH_ROWS]
        outputs = lc.replay_layers(weights, first_layer, 0)
        replay_target_chunks.append(torch.stack([output[:, -1, :] for output in outputs], dim=1).cpu().numpy())
        replay_mean_chunks.append(torch.stack([output[:, :-1, :].mean(dim=1) for output in outputs], dim=1).cpu().numpy())
        replay_prob_chunks.append(lc.death_probability(continuation, outputs[-1]).cpu().numpy())
replayed_target = np.concatenate(replay_target_chunks)
replayed_mean = np.concatenate(replay_mean_chunks)
replayed_probability = np.concatenate(replay_prob_chunks)

target_gap = np.abs(replayed_target - eval_target).max(axis=(0, 2))
mean_gap = np.abs(replayed_mean - eval_feature_mean).max(axis=(0, 2))
parity_by_layer = pd.DataFrame({"layer": range(LAYER_COUNT), "target_max_abs_gap": target_gap,
                                "feature_mean_max_abs_gap": mean_gap})
print(parity_by_layer.to_string(index=False))
layers_match = (np.allclose(replayed_target, eval_target, **FORWARD_TOLERANCE)
                and np.allclose(replayed_mean, eval_feature_mean, **FORWARD_TOLERANCE))
record_check("replay_matches_cached_layers", float(max(target_gap.max(), mean_gap.max())),
             "allclose rtol 1e-4, atol 1e-5, every layer 0..11", layers_match, scaling=SCALING)
probability_gap = float(np.abs(replayed_probability - eval_probability).max())
record_check("replay_probability_matches_cached", probability_gap,
             "allclose rtol 1e-4, atol 1e-5", np.allclose(replayed_probability, eval_probability, **FORWARD_TOLERANCE),
             scaling=SCALING)

fd_report = lc.layer_fd_check(weights, continuation, eval_capture["layer0"],
                              rows=np.arange(FD_ROWS), seed=FD_DIRECTION_SEED)
fd_max_by_layer = [round(value, 9) for value in fd_report["max_abs_error"]]
record_check("finite_difference_every_layer", max(fd_max_by_layer),
             "central difference, step 1e-2, float64, allclose rtol 1e-3, atol 1e-5; 8 rows per layer",
             fd_report["passed"], scaling=SCALING)
print("FD max abs error by layer:", fd_max_by_layer)
STEP_SECONDS["replay_checks"] = round(time.monotonic() - step_start, 2)

 layer  target_max_abs_gap  feature_mean_max_abs_gap
     0            0.000000                  0.000000
     1            0.000003                  0.000002
     2            0.000003                  0.000002
     3            0.000003                  0.000002
     4            0.000003                  0.000004
     5            0.000005                  0.000003
     6            0.000007                  0.000002
     7            0.000007                  0.000004
     8            0.000010                  0.000010
     9            0.000007                  0.000009
    10            0.000007                  0.000006
    11            0.000010                  0.000006


FD max abs error by layer: [4.342e-06, 2.959e-06, 4.979e-06, 3.208e-06, 5.564e-06, 5.162e-06, 3.235e-06, 4.086e-06, 4.259e-06, 5.889e-06, 5.354e-06, 6.384e-06]


## 8. Model Unchanged After Replay
The cached model must be the same object as before the replay (identity hash of the fitted context and cached decoder parameters). Re-scoring the first evaluation rows must reproduce the captured probabilities.

In [8]:
identity_after = pilot.predictor_context_identity(classifier)
record_check("predictor_identity_unchanged_by_replay", identity_after == identity_before,
             "predictor_context_identity equal before and after replay", identity_after == identity_before,
             scaling=SCALING)
recheck_rows = eval_idx[:FD_ROWS]
recheck = probe.capture_flchain_batches(fitted, X[recheck_rows], years[recheck_rows], batch_size=FD_ROWS)
recheck_gap = float(np.abs(recheck["probabilities"][:, 1] - eval_probability[:FD_ROWS]).max())
record_check("predictions_unchanged_after_replay", recheck_gap,
             "allclose rtol 1e-4, atol 1e-5 against the captured probabilities",
             np.allclose(recheck["probabilities"][:, 1], eval_probability[:FD_ROWS], **FORWARD_TOLERANCE),
             scaling=SCALING)
print(f"Identity unchanged: {identity_after == identity_before} | re-scoring gap: {recheck_gap:.2e}")

Identity unchanged: True | re-scoring gap: 4.68e-06


## 9. Parity With Work 6b at Layer 11
The Work 6b archive stores the per-row layer-11 embeddings, gradients and probabilities of the same evaluation rows, in the same order. The chronological split indices must also match. Each per-row comparison is recorded with its tolerance in `parity_layer_11.csv`.

In [9]:
step_start = time.monotonic()
parity = np.load(WORK6B_DIR / "splits_and_predictions.npz")
split_matches_work6b = (
    np.array_equal(parity["h1095_context_idx"], context_idx)
    and np.array_equal(parity["h1095_ref_fit_idx"], ref_fit_idx)
    and np.array_equal(parity["h1095_ref_eval_idx"], ref_eval_idx)
    and np.array_equal(parity["h1095_future_idx"], future_idx)
    and np.array_equal(parity["h1095_y"], y)
    and np.array_equal(parity["h1095_subject_id"], subject_ids)
)
record_check("split_indices_and_labels_match_work6b", int(split_matches_work6b),
             "context, ref_fit, ref_eval, future indices, labels and subject ids equal", split_matches_work6b,
             scaling=SCALING)

six_position = {int(row): position for position, row in enumerate(parity["h1095_eval_idx"])}
six_rows = np.array([six_position[int(row)] for row in eval_idx], dtype=np.int64)
parity_rows = []


def compare_arrays(metric, ours, saved, tolerance):
    difference = np.abs(ours.astype(np.float64) - saved.astype(np.float64))
    passed = bool(np.allclose(ours, saved, **tolerance))
    parity_rows.append({"source": "work6b per-row arrays", "stratum": "all_evaluation_rows", "metric": metric,
                        "rows": len(saved), "saved_value": np.nan, "measured_value": np.nan,
                        "max_abs_difference": float(difference.max()), "rtol": tolerance["rtol"],
                        "atol": tolerance["atol"], "passed": passed})
    record_check(f"{metric}_matches_work6b", float(difference.max()),
                 f"allclose rtol {tolerance['rtol']}, atol {tolerance['atol']}", passed, scaling=SCALING)


compare_arrays("target_embedding_layer_11", eval_target[:, LAST_LAYER], parity["h1095_consistent_emb_actual"][six_rows], FORWARD_TOLERANCE)
compare_arrays("target_gradient_layer_11", eval_gradients[:, LAST_LAYER], parity["h1095_consistent_grad_actual"][six_rows], GRADIENT_TOLERANCE)
compare_arrays("probability_layer_11", eval_probability, parity["h1095_consistent_p_actual"][six_rows], FORWARD_TOLERANCE)
STEP_SECONDS["parity_arrays"] = round(time.monotonic() - step_start, 2)

## 10. Layer Geometry Per Stratum and Summary Parity With Work 6b
For each stratum, the layer geometry uses the target token (spectrum, gradient coherence and dimension, random-direction shares) and the feature-mean token (spectrum). Standardization uses the `ref_fit` embeddings of each layer. At layer 11, the summary values are compared with the Work 6b CSVs. That comparison needs the full stratum, so it is recorded only when the stratum is not limited.

In [10]:
step_start = time.monotonic()
target_scales = [probe.standardization_scale(ref_fit_target[:, layer])[0] for layer in range(LAYER_COUNT)]
mean_scales = [probe.standardization_scale(ref_fit_feature_mean[:, layer])[0] for layer in range(LAYER_COUNT)]

geometry_frames = []
for name, rows in stratum_rows.items():
    frame = lc.layer_geometry_table(
        eval_target[rows], eval_gradients[rows], eval_probability[rows], ref_fit_target,
        feature_mean=eval_feature_mean[rows], scale_feature_mean=ref_fit_feature_mean,
    )
    frame.insert(0, "stratum", name)
    frame.insert(0, "scaling", SCALING)
    frame.insert(0, "build", BUILD)
    geometry_frames.append(frame)
layer_geometry = pd.concat(geometry_frames, ignore_index=True)
layer_geometry.to_csv(OUT_DIR / "layer_geometry.csv", index=False)

SUMMARY_SOURCES = (
    ("embedding_spectrum.csv", "spectrum", None, ("embedding_pr", "top1_var_share", "top5_var_share", "top10_var_share")),
    ("gradient_coherence.csv", "coherence", "probability", ("resultant_length",)),
    ("gradient_dimension.csv", "dimension", "probability", ("centered_pr", "uncentered_pr")),
)
summary_rows = []
for name, rows in stratum_rows.items():
    if len(strata_idx[name]) != full_stratum_sizes[name]:
        print(f"Summary parity with Work 6b skipped for {name}: stratum limited to {len(strata_idx[name])} rows")
        continue
    at_layer = layer_geometry[(layer_geometry["stratum"] == name) & (layer_geometry["token"] == "target")
                              & (layer_geometry["layer"] == LAST_LAYER)]
    for file_name, section, gradient_scale, metrics in SUMMARY_SOURCES:
        saved_table = pd.read_csv(WORK6B_DIR / file_name)
        saved_table = saved_table[(saved_table["target"] == BUILD) & (saved_table["scaling"] == SCALING)
                                  & (saved_table["stratum"] == name)]
        if gradient_scale is not None:
            saved_table = saved_table[saved_table["gradient_scale"] == gradient_scale]
        for _, saved_row in saved_table.iterrows():
            key = saved_row["chart"] if gradient_scale is None else f"{saved_row['chart']}|{gradient_scale}"
            for metric in metrics:
                ours = float(at_layer[(at_layer["section"] == section) & (at_layer["key"] == key)
                                      & (at_layer["metric"] == metric)]["value"].iloc[0])
                saved = float(saved_row[metric])
                summary_rows.append({"source": f"work6b {file_name}", "stratum": name,
                                     "metric": f"{section}|{key}|{metric}", "rows": full_stratum_sizes[name],
                                     "saved_value": saved, "measured_value": ours,
                                     "max_abs_difference": abs(ours - saved),
                                     "rtol": FORWARD_TOLERANCE["rtol"], "atol": FORWARD_TOLERANCE["atol"],
                                     "passed": bool(abs(ours - saved) <= FORWARD_TOLERANCE["atol"] + FORWARD_TOLERANCE["rtol"] * abs(saved))})
for name in full_stratum_sizes:
    if len(strata_idx[name]) != full_stratum_sizes[name]:
        continue
    stratum_summary = [row for row in summary_rows if row["stratum"] == name]
    record_check(f"summary_layer_11_matches_work6b_{name}", max(row["max_abs_difference"] for row in stratum_summary),
                 "allclose rtol 1e-4, atol 1e-5 for spectrum, coherence and dimension values",
                 all(row["passed"] for row in stratum_summary), scaling=SCALING)
pd.DataFrame(parity_rows + summary_rows).to_csv(OUT_DIR / "parity_layer_11.csv", index=False)
checks_so_far = write_checks_and_stop_on_failure()
print(f"Checks passed so far: {int(checks_so_far['passed'].sum())}/{len(checks_so_far)}")
STEP_SECONDS["geometry_and_summary_parity"] = round(time.monotonic() - step_start, 2)

Checks passed so far: 16/16


## 11. Joint Patient Bootstrap Curves
One patient-level bootstrap over the fitting and evaluation patients. Each draw refits every layer's probe on the weighted fitting rows (regularization chosen once on the unweighted fitting rows) and recomputes the geometry on the weighted evaluation rows with the scales fixed. The output has estimates, 95% percentile intervals and the paired difference from layer 11.

In [11]:
step_start = time.monotonic()
fit_payload = {"patient_ids": subject_ids[ref_fit_idx], "y": y[ref_fit_idx],
               "target": ref_fit_target, "feature_mean": ref_fit_feature_mean}
evaluation_payload = {}
for name, idx in strata_idx.items():
    rows = stratum_rows[name]
    evaluation_payload[name] = {"patient_ids": subject_ids[idx], "y": y[idx],
                                "probabilities": eval_probability[rows], "gradients": eval_gradients[rows],
                                "target": eval_target[rows], "feature_mean": eval_feature_mean[rows]}
curves = lc.bootstrap_layer_curves(fit_payload, evaluation_payload, ref_fit_target, ref_fit_feature_mean,
                                   geometry_table=layer_geometry, draws=BOOTSTRAP_DRAWS, seed=BOOTSTRAP_SEED)
layer_curves = curves["layer_curves"]
layer_curves.insert(0, "scaling", SCALING)
layer_curves.insert(0, "build", BUILD)
probe_fits = curves["probe_fits"]
probe_fits.insert(0, "scaling", SCALING)
probe_fits.insert(0, "build", BUILD)
layer_curves.to_csv(OUT_DIR / "layer_curves.csv", index=False)
probe_fits.to_csv(OUT_DIR / "probe_fits.csv", index=False)
infinite_estimates = int(np.isinf(layer_curves["estimate"].astype(float)).sum())
record_check("curve_estimates_have_no_infinite_values", infinite_estimates,
             "no +inf or -inf among estimates (NaN counted in run record)", infinite_estimates == 0, scaling=SCALING)
print(probe_fits.to_string(index=False))
STEP_SECONDS["bootstrap"] = round(time.monotonic() - step_start, 2)

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

... (27 outputs left out)

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


        build    scaling   stratum  layer        token        C C_at_grid_edge  fit_rows  fit_deaths  eval_rows  eval_deaths    auroc
horizon_1095d consistent 1997_eval    0.0       target 0.046416          False       340          27        340           26 0.804018
horizon_1095d consistent 1998-2003    0.0       target 0.046416          False       340          27       1673          120 0.574823
horizon_1095d consistent 1997_eval    0.0 feature_mean 0.359381          False       340          27        340           26 0.742283
horizon_1095d consistent 1998-2003    0.0 feature_mean 0.359381          False       340          27       1673          120 0.531466
horizon_1095d consistent 1997_eval    1.0       target 0.046416          False       340          27        340           26 0.803895
horizon_1095d consistent 1998-2003    1.0       target 0.046416          False       340          27       1673          120 0.502120
horizon_1095d consistent 1997_eval    1.0 feature_mean 0.04641

## 12. Check Gate
Every engineering check must pass before the layer arrays and figures are written. A failed check stops the notebook here.

In [12]:
checks_table = write_checks_and_stop_on_failure()
print(checks_table[["check", "measured", "passed"]].to_string(index=False))
print(f"All {len(checks_table)} checks passed: {bool(checks_table['passed'].all())}")

                                    check                                                         measured  passed
        flchain_csv_sha256_matches_work6b 6b9198b58354c2f2d45edaf4b38c237b9b9fcc22703a73baab75dd6f3d72606c    True
         chronological_split_is_partition                                                             7743    True
 context_fitting_evaluation_rows_disjoint                                                                0    True
    context_rows_used_without_subsampling                                                             5390    True
                capture_shapes_all_layers                                                               12    True
             replay_matches_cached_layers                                                          0.00001    True
        replay_probability_matches_cached                                                         0.000002    True
            finite_difference_every_layer                                       

## 13. Per-Row Arrays
Save the evaluation rows (ids, probabilities, target and feature-mean embeddings and target gradients at every layer) and the per-layer standardization scales, so that the curves can be recomputed without a new fit.

In [13]:
step_start = time.monotonic()
np.savez_compressed(
    OUT_DIR / "layer_arrays.npz",
    eval_row_index=eval_idx,
    eval_stratum=np.concatenate([np.repeat(name, len(idx)) for name, idx in strata_idx.items()]),
    subject_id=subject_ids[eval_idx],
    probability=eval_probability,
    target_embedding=eval_target,
    feature_mean_embedding=eval_feature_mean,
    target_gradient=eval_gradients,
    scale_target=np.stack(target_scales),
    scale_feature_mean=np.stack(mean_scales),
)
STEP_SECONDS["arrays"] = round(time.monotonic() - step_start, 2)

## 14. Figures Across Layers
One figure per metric family. Each panel is one stratum; lines are the estimates with their 95% intervals; the dotted vertical line marks layer 11. The probe figure also draws the model's own AUROC on the same rows and draws.

In [14]:
step_start = time.monotonic()
layer_curves["layer"] = layer_curves["layer"].astype(float)
STRATA_FOR_FIGURES = ("1997_eval", "1998-2003")


def curve_part(stratum, token, chart, metric):
    part = layer_curves[(layer_curves["stratum"] == stratum) & (layer_curves["token"] == token)
                        & (layer_curves["chart"] == chart) & (layer_curves["metric"] == metric)]
    return part.sort_values("layer")


def plot_layer_families(specs, title, file_name, y_label, baseline_metric=None):
    figure, axes = plt.subplots(1, len(STRATA_FOR_FIGURES), figsize=(11, 4), sharey=True, constrained_layout=True)
    for axis, stratum in zip(axes, STRATA_FOR_FIGURES):
        for label, token, chart, metric in specs:
            part = curve_part(stratum, token, chart, metric)
            axis.plot(part["layer"], part["estimate"], marker="o", label=label)
            axis.fill_between(part["layer"], part["low"], part["high"], alpha=0.2)
        if baseline_metric is not None:
            baseline = layer_curves[(layer_curves["stratum"] == stratum) & (layer_curves["metric"] == baseline_metric)].iloc[0]
            axis.axhline(baseline["estimate"], color="black", linestyle="--", label="model death AUROC")
            axis.axhspan(baseline["low"], baseline["high"], color="gray", alpha=0.15)
        axis.axvline(LAST_LAYER, color="gray", linestyle=":")
        axis.set_title(stratum)
        axis.set_xlabel("layer")
    axes[0].set_ylabel(y_label)
    axes[0].legend(fontsize=8)
    figure.suptitle(title)
    figure.savefig(FIG_DIR / file_name, dpi=120)
    plt.close(figure)


plot_layer_families(
    [("target token", "target", "none", "probe_auroc"), ("feature mean", "feature_mean", "none", "probe_auroc")],
    "Linear probe AUROC by layer (95% intervals)", "probe_auroc_by_layer.png", "AUROC", baseline_metric="death_auroc")

for metric in ("embedding_pr", "top1_var_share", "top5_var_share", "top10_var_share"):
    specs = [(f"{token_label} {chart}", token, chart, metric)
             for token, token_label in (("target", "target"), ("feature_mean", "feature mean"))
             for chart in ("raw", "standardized")]
    plot_layer_families(specs, f"{metric} by layer (95% intervals)", f"spectrum_{metric}.png", metric)

for gradient_name in ("probability", "log_odds"):
    for metric in ("resultant_length", "centered_pr", "uncentered_pr"):
        metric_key = f"{gradient_name}_{metric}"
        specs = [(chart, "target", chart, metric_key) for chart in ("raw", "standardized")]
        plot_layer_families(specs, f"{metric_key} by layer (95% intervals)", f"gradient_{metric_key}.png", metric_key)

figure, axes = plt.subplots(1, len(STRATA_FOR_FIGURES), figsize=(11, 4), sharey=True, constrained_layout=True)
for axis, stratum in zip(axes, STRATA_FOR_FIGURES):
    for family in geometry.DIRECTION_FAMILIES:
        part = curve_part(stratum, "target", "raw", f"random_{family}_interior_share")
        spread = curve_part(stratum, "target", "raw", f"random_{family}_interior_se")
        axis.errorbar(part["layer"], part["estimate"], yerr=spread["estimate"], marker="o", capsize=2, label=family)
    axis.axvline(LAST_LAYER, color="gray", linestyle=":")
    axis.set_title(stratum)
    axis.set_xlabel("layer")
axes[0].set_ylabel("interior share (+/- 1 SE of directions)")
axes[0].legend(fontsize=8)
figure.suptitle("Random-direction interior share by layer (raw chart)")
figure.savefig(FIG_DIR / "random_direction_interior_share.png", dpi=120)
plt.close(figure)
print(sorted(path.name for path in FIG_DIR.glob("*.png")))
STEP_SECONDS["figures"] = round(time.monotonic() - step_start, 2)

['gradient_log_odds_centered_pr.png', 'gradient_log_odds_resultant_length.png', 'gradient_log_odds_uncentered_pr.png', 'gradient_probability_centered_pr.png', 'gradient_probability_resultant_length.png', 'gradient_probability_uncentered_pr.png', 'probe_auroc_by_layer.png', 'random_direction_interior_share.png', 'spectrum_embedding_pr.png', 'spectrum_top10_var_share.png', 'spectrum_top1_var_share.png', 'spectrum_top5_var_share.png']


## 15. Run Record
Times, hardware, versions, code and input hashes, row counts, per-step seconds, peak memory and the check total. Nothing here is a conclusion.

In [15]:
git_head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
git_uncommitted = subprocess.check_output(["git", "status", "--porcelain"], text=True).splitlines()
run_record = {
    "started_utc": STARTED_UTC,
    "finished_utc": datetime.now(timezone.utc).isoformat(),
    "runtime_seconds": round(time.monotonic() - T0, 2),
    "step_seconds": STEP_SECONDS,
    "device": DEVICE,
    "gpu_name": GPU_NAME,
    "cuda_version": torch.version.cuda,
    "torch_threads": torch.get_num_threads(),
    "bootstrap_blas_thread_limit": lc.BOOTSTRAP_THREAD_LIMIT,
    "thread_environment": {name: os.environ.get(name) for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")},
    "peak_gpu_memory_mib": round(torch.cuda.max_memory_allocated() / 2**20, 1),
    "peak_rss_mib": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1),
    "git_head": git_head,
    "git_uncommitted": git_uncommitted,
    "sha256": {
        "flchain_layer_compression.ipynb": probe.file_sha256("flchain_layer_compression.ipynb"),
        "layer_compression.py": probe.file_sha256("layer_compression.py"),
        "flchain_probe.py": probe.file_sha256("flchain_probe.py"),
        "prospective_audit.py": probe.file_sha256("prospective_audit.py"),
        "tcav_geometry.py": probe.file_sha256("tcav_geometry.py"),
        "concept_subgroup_drift_pilot.py": probe.file_sha256("concept_subgroup_drift_pilot.py"),
        "fit_checkpoint": fitted["checkpoint_sha256"],
        "flchain_csv": csv_sha256,
        "work6b_splits_and_predictions_npz": probe.file_sha256(WORK6B_DIR / "splits_and_predictions.npz"),
    },
    "versions": {name: metadata.version(name) for name in ("torch", "drift-resilient-tabpfn", "scikit-learn", "numpy", "pandas")},
    "token_count": TOKEN_COUNT,
    "context_rows": int(fitted["context_rows"]),
    "row_counts": row_counts.to_dict(orient="records"),
    "full_stratum_sizes": full_stratum_sizes,
    "evaluation_rows_per_stratum_limit": EVAL_ROWS_PER_STRATUM,
    "bootstrap_draws": BOOTSTRAP_DRAWS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "direction_seed": DIRECTION_SEED,
    "fd_direction_seed": FD_DIRECTION_SEED,
    "fd_rows_per_layer": FD_ROWS,
    "fd_max_abs_error_by_layer": fd_max_by_layer,
    "standardization_zero_columns_target": [int(probe.standardization_scale(ref_fit_target[:, layer])[1]) for layer in range(LAYER_COUNT)],
    "nan_curve_estimates": int(layer_curves["estimate"].astype(float).isna().sum()),
    "checks_passed": int(checks_table["passed"].sum()),
    "checks_total": int(len(checks_table)),
}
(OUT_DIR / "run_record.json").write_text(json.dumps(run_record, indent=2, default=str))
print(f"Run completed in {run_record['runtime_seconds']} s; checks passed {run_record['checks_passed']}/{run_record['checks_total']}")

Run completed in 720.8 s; checks passed 17/17


## Descriptive reading

Written after the run (2026-11-26). Descriptive only: no layer is selected here, and no cutoff is applied. Numbers are estimate [2.5%, 97.5%] from the joint patient bootstrap (500 draws). All checks passed (17/17). Layer-11 parity with Work 6b: per-row embedding, gradient and probability max abs difference ≤ 1.5e-7; summary rows ≤ 1.2e-7.

**Pooled strata (pre-declared).**
- **1997_eval: one sample year, 340 rows / 26 deaths.**
  - Raw embedding PR is 4.07 [3.18, 4.40] at layer 0, 1.78 at layer 3, 2.4–2.9 over layers 5–9 and 2.51 [2.26, 2.79] at layer 11.
  - Probe AUROC is 0.75–0.83 at every layer. The paired difference from layer 11 includes 0 at layers 7–10, for example layer 10 at −0.005 [−0.022, 0.011]. The model's own AUROC is 0.841.
- **1998–2003: pooled sample years, 1,673 / 120.**
  - Raw PR is 1.10 at layer 0, rises to 4.52 [4.37, 4.64] at layer 9 and ends at 2.92 at layer 11.
  - Probe AUROC is 0.50–0.71 at layers 0–6. From layer 7 on it is 0.84–0.86, against the model's 0.865; the paired-difference intervals at layers 7–10 reach 0.
- **Gradient.** R is the resultant length of the probability gradients (1 = all rows point the same way). In both strata, R rises from 0.80–0.82 at layer 0 to about 0.93 at layers 7–10, then 0.975–0.978 at layer 11. The random isotropic interior share falls from about 0.60–0.65 to 0.34–0.35 at layer 9, 0.31–0.33 at layer 10 and 0.17 at layer 11.

**By sample year** (`layer_compression_by_year.ipynb`, `by_year/`; done after the run, outside the pre-declared plan).
- **Embedding PR by year.**
  - The near-1 layer-0 PR of the pooled 1998–2003 stratum appears only when years are pooled. For 1998, 1999, 2000 and 2001 alone the layer-0 PR is 3.6–4.4, as in 1997.
  - The `2002-2003` group itself pools two sample years (grouped as in Work 6b) and also has layer-0 PR 1.00. In the final review's split, not in the saved CSVs, its leading direction separates the two years (|r| 1.00), and each year alone has PR 3.35 (2002) and 3.76 (2003). So this group shows the same year mixing.
  - Within each year group, raw PR at layer 9 is above layer 11 (+0.43 to +1.04; every interval excludes 0). Layer 10 is at or below layer 11 in 5 of 6 groups; 1997_eval gives −0.198 [−0.263, −0.102]. Layer 3 is the lowest middle layer (1.56–2.32).
- **Gradient by year.** Gradients at layers 9–10 are less concentrated than at layer 11 in every group: R about 0.93 vs 0.975, and 0.964–0.968 vs 0.992 in 2002–2003.

**What this shows, descriptively.**
- This dense 7-variable cohort repeats the SUS pattern. Gradient concentration and random-direction TCAV saturation build up gradually, with a final step at layer 11.
- Within a sample year, layer 9 holds more effective dimensions than layer 11; layer 10 does not.
- On future years, the pooled linear probe at layers 7–10 is within the paired-difference interval of layer 11.
- Choosing a layer is left to the user.

**Caveats.**
- **Partial derivative.** Gradients are partial derivatives w.r.t. the target token, with the other tokens fixed.
- **Mixed target token at layer 0.** The layer-0 target token has already been mixed by feature attention. Tokens lie on a sphere of norm sqrt(192).
- **Small fitting set.** The probe is fit on 340 ref_fit rows with 27 deaths, so intervals are wide. C sits at the lower edge of the CV grid (1e-4) at layers 9–11 (`probe_fits.csv`).
- **Probe shift on future years.** The probe and the standardization scale come from 1997 rows. Low probe AUROC on future years at early layers can come from the year shift of the embedding, not only from missing information. The per-year probe was not computed.
- **Padded tokens.** The feature mean includes padded tokens (T = 19).
- **Two-year group.** The `2002-2003` group mixes two sample years (see above).
- **GPU margin.** Peak GPU memory was 7.0 of 8 GiB.